# CIFAR-100 SFL Experiment Baseline Evaluations

This notebook runs experiments for `sfl`, `sfl_gold`, and `centinel` baselines on the **CIFAR100** dataset.

**Configurations:**
- Pretrained Weights: DEFAULT
- Partitions: IID, Non-IID (Dirichlet $\alpha=0.5$)
- Attacks: backdoor, pairflip
- Malicious Fraction: 0.0, 0.7

In [ ]:
import torch
import copy
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import trange
from torch.utils.data import DataLoader, Subset

from src.data.datasets import get_datasets
from src.data.partition import partition_data_iid, partition_data_dirichlet
from src.data.poisoned_dataset import PoisonedDataset
from src.models.split import get_split_models
from src.core.client import SplitFedClient
from src.core.server import SplitFedServer
from src.core.seed import set_seed
from src.algorithms import run_sfl_round, run_sfl_gold_round
from src.algorithms.centinel import run_sfl_centinel_round, CentinelState, compute_centroids
from src.algorithms.evaluate import (
    evaluate_accuracy, 
    evaluate_backdoor_asr, 
    evaluate_targeted_asr, 
    evaluate_pair_flip_asr
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# EXPERIMENTAL CONFIGURATION
# ==========================================

DATASET = "CIFAR100"
PRETRAINED_WEIGHTS = "DEFAULT"

MALICIOUS_FRACTIONS = [0.0, 0.7]
ATTACK_TYPES = ["backdoor", "pair_flip"]
PARTITIONS = ["iid", "non-iid"]
DIRICHLET_ALPHA = 0.5

# Fixed Training Hyperparameters
NUM_CLIENTS = 10
BATCH_SIZE = 64
LEARNING_RATE = 3e-3
ROUNDS = 50
GLOBAL_SEED = 42

set_seed(GLOBAL_SEED)

# Attack arguments matching typical CIFAR100 attacks
ATTACK_KWARGS = {
    "backdoor_poison_fraction": 1.0,
    "backdoor_target_label": 0,
    "backdoor_source_labels": [1, 2, 3],
    "trigger_size": 3,
    "trigger_value_raw": 1.0,
    "trigger_pos": "br",
    "flip_fraction": 1.0,
    "label_pairs_to_flip": [(1, 8), (2, 7), (3, 9)]
}

# Centinel Hyperparameters
NUM_REF_SAMPLES_PER_LABEL = 10
TAU = 0.1
OMEGA = 0.7
Q_I = 0.8
RHO = 0.4
ETA = 0.6
KAPPA = 0.7
ZETA = 0.3

# ==========================================
# DATA PREPARATION
# ==========================================

print(f"Loading {DATASET} dataset...")
train_data, test_data = get_datasets(dataset_name=DATASET, hf_token=None)
test_loader = DataLoader(
    test_data, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

def build_ref_loader(test_data):
    labels = np.array(test_data.hf_dataset[test_data.label_key])
    ref_indices = []
    num_classes = len(np.unique(labels))
    for c in range(num_classes):
        c_indices = np.where(labels == c)[0]
        k = min(NUM_REF_SAMPLES_PER_LABEL, len(c_indices))
        if k > 0:
            ref_indices.extend([int(x) for x in np.random.choice(c_indices, k, replace=False)])
    ref_dataset = Subset(test_data, ref_indices)
    return DataLoader(ref_dataset, batch_size=BATCH_SIZE, shuffle=False)

ref_loader = build_ref_loader(test_data)


In [ ]:
def run_experiment(algorithm, malicious_fraction, attack_type, partition_method, dirichlet_alpha=0.5):
    
    # 1. Partition Data
    if partition_method == "iid":
        client_datasets = partition_data_iid(train_data, NUM_CLIENTS)
    else:
        client_datasets = partition_data_dirichlet(train_data, NUM_CLIENTS, alpha=dirichlet_alpha)
        
    # 2. Determine malicious clients
    num_malicious = int(NUM_CLIENTS * malicious_fraction)
    malicious_clients_indices = set(np.random.choice(NUM_CLIENTS, num_malicious, replace=False))
    
    # 3. Model Initialization
    base_client_model, server_model = get_split_models(DATASET, weights=PRETRAINED_WEIGHTS)
    server = SplitFedServer(model=server_model, num_clients=NUM_CLIENTS, lr=LEARNING_RATE, device=device)
    
    clients = []
    for i in range(NUM_CLIENTS):
        client_model = copy.deepcopy(base_client_model)
        is_mal = i in malicious_clients_indices
        c_dataset = client_datasets[i]
        if is_mal and attack_type != 'none':
            c_dataset = PoisonedDataset(
                c_dataset, 
                attack_type=attack_type, 
                attack_kwargs=ATTACK_KWARGS, 
                dataset_name=DATASET, 
                seed=GLOBAL_SEED+i
            )
        client = SplitFedClient(
            client_id=i, 
            model=client_model, 
            dataset=c_dataset, 
            batch_size=BATCH_SIZE, 
            lr=LEARNING_RATE, 
            device=device, 
            is_malicious=is_mal
        )
        clients.append(client)
        
    # 4. Centinel Initialization (if applicable)
    state = None
    if algorithm == "centinel":
        state = CentinelState(NUM_CLIENTS, tau=TAU, omega=OMEGA, Q_i=Q_I, rho=RHO, eta=ETA, kappa=KAPPA, zeta=ZETA)
        client0 = clients[0]
        client0.model.eval()
        with torch.no_grad():
            for data, target in ref_loader:
                acts = client0.model(data.to(device))
                c, counts = compute_centroids(acts, target.to(device))
                for lbl in c:
                    if lbl in state.global_centroids:
                        state.global_centroids[lbl] = (state.global_centroids[lbl] + c[lbl])/2
                    else:
                        state.global_centroids[lbl] = c[lbl]
                        
    # 5. Training Loop
    historical_train_acc = []
    historical_test_acc = []
    historical_asr = []
    
    pbar = trange(ROUNDS, desc="Training", unit="round")
    for r in pbar:
        if algorithm == "sfl":
            train_loss, train_acc = run_sfl_round(clients, server, local_epochs=1)
        elif algorithm == "sfl_gold":
            train_loss, train_acc = run_sfl_gold_round(clients, server, local_epochs=1)
        elif algorithm == "centinel":
            train_loss, train_acc, scores, accepted_clients = run_sfl_centinel_round(
                clients, server, state, ref_loader, local_epochs=1, device=device
            )
        
        # Evaluate Global Models
        eval_client = clients[0].model
        test_acc = evaluate_accuracy(eval_client, server.model, test_loader, device)
        
        asr = 0.0
        if attack_type != 'none':
            if attack_type == 'backdoor':
                asr = evaluate_backdoor_asr(eval_client, server.model, test_loader, 
                                            ATTACK_KWARGS.get('backdoor_source_labels', []), 
                                            ATTACK_KWARGS.get('backdoor_target_label', 0), 
                                            ATTACK_KWARGS, device)
            elif attack_type == 'pair_flip':
                asr = evaluate_pair_flip_asr(eval_client, server.model, test_loader, 
                                             ATTACK_KWARGS.get('label_pairs_to_flip', []), 
                                             device)
                                             
        historical_train_acc.append(train_acc)
        historical_test_acc.append(test_acc)
        historical_asr.append(asr)
        pbar.set_postfix(loss=f"{train_loss:.4f}", test_acc=f"{test_acc:.4f}", asr=f"{asr:.4f}")
        
    return {
        "train_acc": historical_train_acc,
        "test_acc": historical_test_acc,
        "asr": historical_asr
    }


In [ ]:
results = {}

for partition in PARTITIONS:
    for attack in ATTACK_TYPES:
        for mal_frac in MALICIOUS_FRACTIONS:
            
            # If malicious fraction is 0, the attack type is practically 'none'.
            # We run it once for the first attack in ATTACK_TYPES to avoid duplicating the 0.0 experiments.
            if mal_frac == 0.0 and attack != ATTACK_TYPES[0]:
                continue
                
            eff_attack = attack if mal_frac > 0 else 'none'
            
            for algo in ["sfl", "sfl_gold", "centinel"]:
                print(f"\n{'='*60}")
                print(f"Partition: {partition} | Attack: {eff_attack} | Malicious Fraction: {mal_frac} | Algo: {algo}")
                print(f"{'='*60}")
                
                res = run_experiment(
                    algorithm=algo,
                    malicious_fraction=mal_frac,
                    attack_type=eff_attack,
                    partition_method=partition,
                    dirichlet_alpha=DIRICHLET_ALPHA
                )
                
                results[(partition, eff_attack, mal_frac, algo)] = res


In [ ]:
def plot_results():
    # We have multiple subplots for each partition/attack/mal_frac combination
    for partition in PARTITIONS:
        for attack in ATTACK_TYPES:
            for mal_frac in MALICIOUS_FRACTIONS:
                if mal_frac == 0.0 and attack != ATTACK_TYPES[0]:
                    continue
                    
                eff_attack = attack if mal_frac > 0 else 'none'
                
                plt.figure(figsize=(10, 5))
                
                for algo in ["sfl", "sfl_gold", "centinel"]:
                    res = results.get((partition, eff_attack, mal_frac, algo))
                    if res:
                        line, = plt.plot(range(1, ROUNDS+1), res["test_acc"], label=f"{algo} (Test Acc)")
                        if eff_attack != 'none':
                            plt.plot(range(1, ROUNDS+1), res["asr"], color=line.get_color(), linestyle="--", label=f"{algo} (ASR)")
                
                plt.title(f"CIFAR100 | {partition.upper()} | Attack: {eff_attack} | Malicious: {mal_frac*100}%")
                plt.xlabel("Communication Round")
                plt.ylabel("Rate / Accuracy")
                plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                plt.grid(True)
                plt.tight_layout()
                plt.show()

plot_results()


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

def calc_ci(data):
    mean = np.mean(data)
    std = np.std(data, ddof=1) if len(data) > 1 else 0.0
    # t_critical for 95% CI with df=2 (n=3) is 4.303
    t_crit = 4.303
    margin_of_error = t_crit * (std / np.sqrt(len(data))) if std > 0 else 0.0
    if np.isnan(margin_of_error) or margin_of_error == 0.0:
        return f"{mean*100:.2f}%"
    return f"{mean*100:.2f}% ± {margin_of_error*100:.2f}%"

table_data = []
for partition in PARTITIONS:
    for attack in ATTACK_TYPES:
        for mal_frac in MALICIOUS_FRACTIONS:
            eff_attack = attack if mal_frac > 0 else 'none'
            if mal_frac == 0.0 and attack != ATTACK_TYPES[0]:
                continue
            
            for algo in ["sfl", "sfl_gold", "centinel"]:
                res = results.get((partition, eff_attack, mal_frac, algo))
                if res:
                    last_train = res["train_acc"][-3:]
                    last_test = res["test_acc"][-3:]
                    last_asr = res["asr"][-3:]
                    
                    table_data.append({
                        "Partition": partition,
                        "Attack": eff_attack,
                        "Malicious %": f"{mal_frac*100}%",
                        "Algorithm": algo,
                        "Train Acc (95% CI)": calc_ci(last_train),
                        "Test Acc (95% CI)": calc_ci(last_test),
                        "ASR (95% CI)": calc_ci(last_asr)
                    })

# Display the table
df = pd.DataFrame(table_data)
display(df)
